# Athena Research Particle Analysis Testing Notebook

This notebook mirrors the package-level placement of `operations_tests.ipynb`. It keeps reusable particle loading, deposition, and plotting checks outside `problem_plotting`, which should be reserved for problem-specific scripts.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from athena_research.core.io_utils import read_binary
from athena_research.core.particle_io import (
    particle_column_xy,
    particle_profile_x,
    read_particle_binary,
    read_particle_binary_positions,
)

%matplotlib inline

## 1. Select a Run Directory

Change `ROOT` and `CASE` to point at your simulation output.

In [ ]:
ROOT = Path('/path/to/your/simulation/runs')
CASE = 'square_ito'

run_dir = ROOT / 'runs' / CASE
bin_dir = run_dir / 'bin'
pbin_dir = run_dir / 'pbin'

def latest_file(directory, suffix):
    files = sorted(Path(directory).glob(f'*{suffix}'))
    if not files:
        raise FileNotFoundError(f'No *{suffix} files found in {directory}')
    return files[-1]

grid_file = latest_file(bin_dir, '.bin')
particle_file = latest_file(pbin_dir, '.prtclbin')

print(grid_file)
print(particle_file)

## 2. Load Grid and Particle Data

`read_binary` loads the AthenaK grid dump. `read_particle_binary` loads the particle output and any grid quantities sampled into the particle file.

In [ ]:
grid = read_binary(str(grid_file))
particles = read_particle_binary(particle_file)

print('grid time:', grid['time'], 'cycle:', grid['cycle'])
print('particle time:', particles['time'], 'cycle:', particles['ncycle'])
print('particles:', particles['nparticles'])
print('particle keys:', sorted(particles.keys()))

## 3. Generic Deposition Helpers

The reusable helpers in `athena_research.core.particle_io` deposit particle positions onto the same logical grid as the gas.

In [ ]:
positions = read_particle_binary_positions(particle_file)
print('position-only keys:', sorted(positions.keys()))
print('position arrays:', positions['x'].shape, positions['y'].shape, positions['z'].shape)

## 4. Quick Particle Column Plot

In [ ]:
column = particle_column_xy(particles, grid)
extent = [grid['x1min'], grid['x1max'], grid['x2min'], grid['x2max']]

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(column, origin='lower', extent=extent, cmap='magma')
fig.colorbar(im, ax=ax, label='normalized particle column')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'{CASE}: particle column')
fig.tight_layout()

## 5. Square-Wave Tracer Comparison

This reproduces the basic square-pulse comparison across the three implemented tracer methods when all three run directories are available.

In [ ]:
SQUARE_CASES = {
    'classical': ('square_classical', 'Classical', '#3b6fb6'),
    'ito': ('square_ito', 'Ito-2', '#111111'),
    'lagrangian_mc': ('square_lagrangian_mc', 'MC', '#b33c2e'),
}

def load_case(root, case_name):
    run_dir = Path(root) / 'runs' / case_name
    grid_file = latest_file(run_dir / 'bin', '.bin')
    particle_file = latest_file(run_dir / 'pbin', '.prtclbin')
    return read_binary(str(grid_file)), read_particle_binary(particle_file)

loaded = {}
for key, (case_name, label, color) in SQUARE_CASES.items():
    try:
        loaded[key] = (*load_case(ROOT, case_name), label, color)
    except FileNotFoundError as exc:
        print(f'Skipping {case_name}: {exc}')

if loaded:
    ref_grid = next(iter(loaded.values()))[0]
    x = np.linspace(ref_grid['x1min'], ref_grid['x1max'], ref_grid['Nx1'], endpoint=False)
    x = x + 0.5 * (ref_grid['x1max'] - ref_grid['x1min']) / ref_grid['Nx1']

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for grid_i, particles_i, label, color in loaded.values():
        profile = particle_profile_x(particles_i, grid_i)
        ax.step(x, profile, where='mid', label=label, color=color)
    ax.set_xlabel('x')
    ax.set_ylabel('normalized tracer density')
    ax.set_title(f'Square-wave tracer comparison, t={ref_grid["time"]:.4g}')
    ax.legend(frameon=False)
    ax.grid(alpha=0.2)
    fig.tight_layout()

## 6. Notes for Large Turbulence Outputs

The 128^3 turbulence runs can contain tens of millions of particles. For those files, use `read_particle_binary_positions()` when plots only need positions, then deposit with the same generic helpers shown above.